## 1 — Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q anthropic chromadb sentence-transformers python-dotenv gradio

## 2 — Imports & Anthropic Client

In [ ]:
import os
import textwrap
from pathlib import Path
from dotenv import load_dotenv
import anthropic
import chromadb
import gradio as gr
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

load_dotenv(override=True)

ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_KEY:
    raise ValueError("Set ANTHROPIC_API_KEY in a .env file or environment variable")


FAST_MODEL  = "claude-haiku-4-5-20251001"
SMART_MODEL = "claude-sonnet-4-6"

claude = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
print(f"✅ Anthropic client ready")
print(f"   Fast model  : {FAST_MODEL}")
print(f"   Smart model : {SMART_MODEL}")

## 3 — Build the SA Car Knowledge Base

Reads `sa_cars.txt` line by line.  
If the file doesn't exist, a rich sample dataset is created automatically.

In [ ]:
SAMPLE_KNOWLEDGE = """
Toyota Hilux is the best-selling vehicle in South Africa and is ideal for gravel and farm roads.
Toyota Fortuner is a popular family SUV in South Africa known for reliability and resale value.
Volkswagen Polo is the top-selling passenger car in South Africa, valued for fuel efficiency.
Ford Ranger is a leading double-cab bakkie popular with both urban and rural South African drivers.
Suzuki Swift and Baleno are highly affordable city cars in South Africa with low running costs.
Hyundai and Kia offer competitive warranties of up to 5 years in South Africa, reducing ownership risk.
South African petrol prices are regulated and updated on the first Wednesday of every month.
93 octane unleaded petrol is the standard fuel grade used inland (Gauteng, Limpopo, Mpumalanga).
95 octane unleaded petrol is required at coastal regions (Western Cape, KwaZulu-Natal, Eastern Cape).
Diesel vehicles are popular in South Africa for long-distance travel due to lower fuel costs per km.
South Africa has some of the highest vehicle insurance premiums in Africa due to high theft rates.
The Western Cape has the best-maintained road infrastructure in South Africa.
Gauteng has the highest concentration of highways and toll roads in South Africa.
Limpopo and Mpumalanga have many unpaved roads requiring higher ground clearance vehicles.
Load shedding (loadshedding) has driven interest in electric vehicles with home charging in South Africa.
Electric vehicles (EVs) are still niche in South Africa due to limited public charging infrastructure.
The NAAMSA vehicle registration data shows bakkies (pickup trucks) account for over 25% of SA new car sales.
South African consumers prefer automatic transmissions increasingly due to urban traffic congestion.
CarTrack and Matrix are leading vehicle tracking providers used in South Africa to reduce theft risk.
Vehicle financing through WesBank, Absa, and Standard Bank is common in South Africa.
The AA (Automobile Association of South Africa) recommends servicing every 15,000 km or annually.
Toyota and Volkswagen dominate the South African new car market with the highest dealer networks.
Second-hand car values in South Africa depreciate steeply in the first 3 years of ownership.
Tyre wear is accelerated on South African roads due to potholes and extreme summer heat.
Catalytic converter theft is a major problem in South Africa, particularly affecting SUVs and bakkies.
Smash-and-grab incidents at traffic lights are common in Johannesburg and Cape Town CBD areas.
Suzuki has the lowest cost of ownership among small cars sold in South Africa according to AA data.
Buying a certified pre-owned (CPO) vehicle from a dealership provides warranty protection in South Africa.
Private vehicle sales through AutoTrader SA are common and often 10-20% cheaper than dealers.
Vehicle roadworthiness testing (roadworthy certificate) is required for ownership transfer in South Africa.
South Africa drives on the left side of the road, same as the UK, Australia, and most of Africa.
The N1, N2, and N3 are South Africa's busiest national highways connecting major cities.
Bakkie (pickup truck) ownership is a cultural status symbol in many South African communities.
Turbocharged engines are increasingly common in South Africa for better fuel economy under load.
Annual licensing fees in South Africa vary by province and vehicle mass, costing R500-R2000 per year.
The South African Rand's weakness makes imported vehicles significantly more expensive than locally assembled ones.
Volkswagen, BMW, and Ford have local assembly plants in South Africa, reducing their vehicle prices.
Mercedes-Benz C-Class is assembled in East London, South Africa, making it competitively priced locally.
WBHO and other construction companies prefer Toyota Land Cruiser for remote South African project sites.
Minibus taxis (Toyota Quantum/HiAce) dominate South African public transport and are high-volume sellers.
Spare parts availability is best for Toyota, VW, and Ford in South Africa due to dealer network size.
Flooding in KwaZulu-Natal has highlighted the need for higher ground clearance in coastal SA provinces.
South African highways have a general speed limit of 120 km/h, with urban limits of 60 km/h.
Drunk driving is a significant road safety problem in South Africa with strict legal blood alcohol limits.
Visibility cameras and dashcams are increasingly popular in South Africa for insurance claim evidence.
The Gauteng freeway improvement project (GFIP) toll roads use e-tag systems for electronic billing.
Fuel-efficient hybrid vehicles like Toyota Corolla Cross hybrid are growing in popularity in South Africa.
South African summer temperatures can exceed 40°C, stressing cooling systems and tyre pressure management.
"Sandton school run" traffic makes large SUVs like Fortuner and Prado popular in Johannesburg northern suburbs.
Cape Town's mountainous terrain makes hill-start assist and strong brakes essential features for residents.

Isuzu D-Max is a popular workhorse bakkie in South Africa used heavily in agriculture and mining sectors.
Nissan Navara offers a comfortable ride for a bakkie and is popular for mixed urban and off-road use in South Africa.
Mitsubishi Triton is a budget-friendly double-cab bakkie option in South Africa with solid off-road capability.
GWM (Great Wall Motors) P-Series bakkie has gained popularity in South Africa for its affordability and features.
Haval SUVs from GWM are rapidly growing market share in South Africa as affordable Chinese alternatives to Japanese brands.
Chery Tiggo and Omoda models are entering the South African market offering competitive pricing against Korean brands.
BYD electric vehicles are officially launched in South Africa making EVs more accessible at lower price points.
The Volvo EX30 and EX40 are the most popular premium electric SUVs available in South Africa.
South Africa's Joburg to Cape Town N1 highway (1,400 km) is one of the most-driven long-distance routes in the country.
The Garden Route along the N2 in the Western Cape is a scenic but challenging coastal drive with sharp bends.
Fog on the Huguenot Tunnel and Du Toitskloof Pass in the Western Cape requires extra caution and good headlights.
The Sani Pass in KwaZulu-Natal is a famous 4x4-only route requiring low-range 4WD and high ground clearance.
Off-road trails like Lesotho Highlands and Kruger bushveld tracks demand skid plates and all-terrain tyres.
Suspension upgrades are commonly fitted to bakkies in South Africa to handle corrugated gravel farm roads.
A canopy (tonneau cover) is a popular bakkie accessory in South Africa for security and load protection.
Tow bars are a common fitting on South African bakkies for trailers, caravans, and horse boxes.
Caravan and camping culture is strong in South Africa making towing capacity a key purchase consideration.
The Jurgens and Sprite caravan brands are popular in South Africa and compatible with most large bakkies and SUVs.
Game reserve driving in Limpopo and Mpumalanga exposes vehicles to dust, sand, and acacia thorns requiring robust tyres.
Spare wheel carriers and high-lift jacks are recommended accessories for South African bush driving.
Run-flat tyres are increasingly specified on luxury German cars sold in South Africa to reduce roadside vulnerability.
Pirelli, Bridgestone, and Goodyear are the most widely available tyre brands in South Africa.
Nitrogen tyre inflation is available at many South African fuel stations and reduces pressure variation in extreme heat.
Wheel and tyre theft from parked vehicles is a growing problem in South African townships and suburbs.
Locking wheel nuts are recommended as a basic security measure for all South African vehicles.
Ghost immobilisers are fitted by many South African vehicle owners to prevent relay key theft on keyless entry cars.
Comprehensive car insurance in South Africa typically costs 1-2% of the vehicle value per month.
Third-party only insurance is a legal minimum in South Africa and is common for older, lower-value vehicles.
OUTsurance, Santam, Discovery Insure, and Hollard are leading car insurance providers in South Africa.
Discovery Insure offers behaviour-based insurance discounts through the Vitality Drive programme in South Africa.
Tracker device installation is often required by South African insurers as a condition of comprehensive cover.
The SA Police Service (SAPS) registers over 60,000 vehicle thefts annually making security a top ownership priority.
Gauteng accounts for the highest number of vehicle hijackings in South Africa, particularly in southern Johannesburg.
Vehicle hijacking hotspots in South Africa include robot (traffic light) stops, driveways, and parking lots.
Armed response vehicle protection and private security estates are popular in South Africa for high-value car owners.
Parking in well-lit, guarded areas is strongly recommended for all vehicles in South African cities after dark.
South African law requires all vehicles to carry a valid licence disc displayed on the windscreen.
Traffic fines in South Africa can be issued for speeding, no seatbelts, using a mobile phone while driving, and rolling stops.
The AARTO demerit system in South Africa can result in licence suspension for repeat traffic offenders.
South African roads have a high incidence of pedestrians, livestock, and potholes requiring constant vigilance.
Livestock on roads is a major hazard particularly at night in the Eastern Cape, Northern Cape, and Free State.
Night driving in rural South Africa is strongly discouraged due to unlit pedestrians, animals, and abandoned vehicles.
High-beam headlights and a bull bar are recommended for rural night driving in South Africa.
South African National Parks charge entrance fees for Kruger, Addo, and other parks requiring a valid vehicle licence.
The South African school holiday seasons (December, Easter, June/July) cause extreme traffic on N1, N2, and N3.
AA Emergency Service provides roadside assistance including towing, battery jump-start, and flat tyre help across South Africa.
Budget Car Rental, Avis, and Europcar are major car hire companies operating at all South African airports.
South African hire car companies often restrict vehicles from crossing borders into neighbouring countries without prior approval.
Petrol station density is low in the Northern Cape and Free State requiring careful fuel planning on long trips.
Many South African petrol stations do not accept foreign credit cards and cash may be required in rural areas.
The Drive.co.za and WeBuyCars platforms are popular for buying and selling used vehicles in South Africa.
WeBuyCars operates physical inspection centres across South Africa providing instant cash offers for used vehicles.
AutoTrader South Africa lists over 60,000 used vehicles at any given time making it the largest local classifieds platform.
Dealerships in South Africa are regulated by the Motor Industry Ombudsman of South Africa (MIOSA) for dispute resolution.
Service plans and maintenance plans sold with new South African vehicles cover scheduled services for 3-5 years.
Extended warranties are available from South African dealerships and third parties like Motorite and Warrantywise.
Fitment centres like Tiger Wheel & Tyre and Hi-Q are national chains offering tyre, brake, and suspension services in South Africa.
Midas and Repco auto parts stores are the most widely available spare parts retailers across South Africa.
SMMEs and small businesses in South Africa commonly use the Toyota Quantum panel van for goods delivery.
Rideshare services Bolt and Uber operate in major South African cities and use Toyota Etios, Polo, and similar models.
South African taxis (metered cabs) are different from minibus taxis and operate in CBD areas of major cities.
The South African government offers a customs duty rebate on electric vehicles to encourage EV adoption.
Solar panel home charging systems for EVs are growing in South Africa as a load-shedding workaround for EV owners.
BMW i3 and i4 are popular urban EVs in South Africa used in Johannesburg and Cape Town where charging infrastructure is better.
Nissan Leaf is the most affordable used electric car available in South Africa with good inner-city range.
The MIDC (Motor Industry Development Council) supports the local assembly of vehicles and components in South Africa.
Ford Silverton plant in Pretoria assembles the Ford Ranger locally reducing its price versus fully imported competitors.
Toyota Prospecton plant in KwaZulu-Natal assembles the Hilux and Fortuner making them more competitively priced in South Africa.
BMW Rosslyn plant near Pretoria assembles the BMW 3 Series locally for both the SA market and export.
""".strip()


kb_path = Path("sa_cars.txt")
if not kb_path.exists():
    kb_path.write_text(SAMPLE_KNOWLEDGE, encoding="utf-8")
    print(f"✅ Created sample knowledge base: {kb_path} ({len(SAMPLE_KNOWLEDGE.splitlines())} facts)")
else:
    print(f"✅ Found existing knowledge base: {kb_path}")

## 4 — Create the ChromaDB Vector Store (RAG)

In [ ]:

print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model ready")


chroma_client = chromadb.PersistentClient(path="sa_cars_db")


try:
    chroma_client.delete_collection("sa_cars")
except Exception:
    pass

collection = chroma_client.get_or_create_collection("sa_cars")


with open("sa_cars.txt", encoding="utf-8") as f:
    facts = [line.strip() for line in f.readlines() if line.strip()]

for i, fact in enumerate(facts):
    embedding = embed_model.encode(fact).tolist()
    collection.add(
        documents=[fact],
        embeddings=[embedding],
        ids=[str(i)]
    )

print(f"✅ Vector store ready — {collection.count()} facts indexed")

## 5 — RAG Retrieval Function

In [ ]:
def retrieve_facts(query: str, n_results: int = 6) -> str:
    """
    Embed the query and retrieve the top-n most relevant facts
    from the SA car knowledge base.
    """
    embedding = embed_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[embedding],
        n_results=n_results
    )
    return "\n".join(results["documents"][0])


# Quick test
print("Test retrieval for 'best bakkie for farm roads':")
print(retrieve_facts("best bakkie for farm roads"))

## 6 — Agent Definitions

Four specialised agents mirror the original notebook's pipeline:

| SA Car Equivalent |
 | **Car Safety Agent** — suitability for SA roads & conditions |
| **SA Market Agent** — specific local models, prices, dealers |
| **Ownership Cost Agent** — fuel, insurance, maintenance |
| Diet Recommendation Agent | **Final Recommendation Agent** — full advice report |

In [ ]:
def llm(prompt: str, model: str = FAST_MODEL) -> str:
    """Simple Claude call — returns the text response."""
    response = claude.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()



def car_safety_agent(query: str) -> str:
    """
    Uses RAG to retrieve relevant SA car facts, then assesses
    which vehicles and features are safe/suitable for the user's needs.
    Mirrors: food_safety_agent()
    """
    context = retrieve_facts(query)

    prompt = f"""
You are a South African automotive safety and road conditions expert.

South African car and road knowledge:
{context}

User question:
{query}

Based on the knowledge above, return:
- Vehicles / features WELL SUITED to South African conditions for this need
- Vehicles / features to AVOID for this need
- Key South African road or safety considerations relevant to the question
"""
    return llm(prompt)



def sa_market_agent(safety_advice: str) -> str:
    """
    Suggests specific South African car models available locally,
    with approximate pricing in ZAR.
    Mirrors: nigerian_food_agent()
    """
    prompt = f"""
You are a South African car dealership expert with knowledge of the local market.

Based on this road safety and suitability advice:
{safety_advice}

Suggest specific South African car models for these three budgets:

Budget (under R300,000)
Mid-range (R300,000 – R600,000)
Premium (above R600,000)

For each suggestion include:
- Model name and variant
- Approximate new price in ZAR
- One key reason it suits the user's needs in South Africa
"""
    return llm(prompt)



def ownership_cost_agent(market_advice: str) -> str:
    """
    Explains the true cost of owning each recommended car in South Africa —
    fuel, insurance, maintenance, licensing.
    Mirrors: nutrition_reasoning_agent()
    """
    prompt = f"""
You are a South African personal finance and motoring cost expert.

Analyse the true monthly and annual cost of owning these recommended vehicles in South Africa:

{market_advice}

For each vehicle cover:
- Estimated monthly fuel cost (assume 1,500 km/month at current SA fuel prices)
- Typical comprehensive insurance premium range (ZAR/month)
- Service plan / maintenance cost
- Annual licensing fee (approximate)
- Overall value-for-money verdict for a South African buyer
"""
    return llm(prompt)



def final_recommendation_agent(query: str, safety: str, market: str, costs: str) -> str:
    """
    Synthesises all three agents into a single, actionable recommendation report.
    Uses the smarter model for the final output.
    Mirrors: diet_recommendation()
    """
    prompt = f"""
You are a trusted South African motoring advisor writing a final recommendation report.

Original user question:
{query}

Road Safety & Suitability Analysis:
{safety}

South African Market Options:
{market}

Ownership Cost Analysis:
{costs}

Write a clear, friendly, well-structured final car recommendation report for a South African buyer.
Use markdown formatting with headers and bullet points.
End with one single TOP PICK recommendation and a one-sentence reason why.
"""
    return llm(prompt, model=SMART_MODEL)


print("✅ All four agents defined")

## 7 — Pipeline Orchestrator

In [ ]:
def car_recommendation(query: str) -> str:
    """
    Full multi-agent pipeline — mirrors diet_recommendation() from the original.

    1. Car Safety Agent   → RAG-backed road & vehicle suitability
    2. SA Market Agent    → local models & ZAR pricing
    3. Ownership Cost Agent → fuel, insurance, maintenance
    4. Final Recommendation → synthesised markdown report
    """
    print("[1/4] Running Car Safety Agent...")
    safety = car_safety_agent(query)

    print("[2/4] Running SA Market Agent...")
    market = sa_market_agent(safety)

    print("[3/4] Running Ownership Cost Agent...")
    costs = ownership_cost_agent(market)

    print("[4/4] Running Final Recommendation Agent...")
    report = final_recommendation_agent(query, safety, market, costs)

    return f"""## 🚗 SA CAR RECOMMENDATION REPORT

---

### Road Safety & Suitability
{safety}

---

### South African Market Options
{market}

---

### Ownership Cost Analysis
{costs}

---

### ✅ Final Recommendation
{report}
"""


print("✅ Pipeline orchestrator ready")

## 8 — Quick Smoke Test

In [ ]:

test_query = "I live in Johannesburg and need a reliable family car for school runs and highway driving"
print("Testing Car Safety Agent...\n")
print(car_safety_agent(test_query))

## 9 — Gradio UI

In [ ]:
THEME = gr.themes.Base(
    primary_hue="green",
    secondary_hue="emerald",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "sans-serif"],
).set(
    body_background_fill="#0d1117",
    body_text_color="#e2e0f0",
    block_background_fill="#161b22",
    block_border_color="#30363d",
    block_title_text_color="#4ade80",
    input_background_fill="#0d1117",
    button_primary_background_fill="linear-gradient(135deg, #16a34a, #15803d)",
    button_primary_text_color="#fff",
)


def run(query: str) -> str:
    """Gradio entry point — wraps the pipeline."""
    if not query or not query.strip():
        return "⚠️ Please enter a question about buying or owning a car in South Africa."
    return car_recommendation(query)


with gr.Blocks(theme=THEME, title="SA Car Advisor") as ui:

    gr.HTML("""
    <div style="text-align:center; padding:1.5rem 0 0.5rem;">
      <h1 style="font-size:2rem; font-weight:800;
                 background:linear-gradient(135deg, #4ade80, #22d3ee);
                 -webkit-background-clip:text; -webkit-text-fill-color:transparent;
                 margin-bottom:6px;">
        🚗 South African Car Advisor
      </h1>
      <p style="color:#8b8aaa; font-size:0.9rem;">
        Multi-agent RAG system · Powered by Anthropic Claude
      </p>
    </div>
    """)

    with gr.Tabs():

        
        with gr.Tab("🔍 Get Advice"):
            gr.Markdown(
                "Ask anything about buying, owning, or choosing a car in South Africa. "
                "The system runs four specialist AI agents to build your personalised report."
            )

            query_box = gr.Textbox(
                label="Your question",
                placeholder="e.g. What is the best car for long road trips from Johannesburg to Cape Town?",
                lines=3
            )

            submit_btn = gr.Button("🚗 Get My Car Recommendation", variant="primary", size="lg")

            gr.Examples(
                examples=[
                    ["I live in Cape Town and need a reliable city car with low fuel costs"],
                    ["What is the best bakkie for a farm in Limpopo under R400,000?"],
                    ["I need a family SUV in Johannesburg for school runs and highway driving"],
                    ["Which car has the lowest total cost of ownership in South Africa?"],
                    ["Best car for driving between Durban and Johannesburg regularly?"],
                ],
                inputs=query_box
            )

            output = gr.Markdown(label="Car Recommendation Report")

            submit_btn.click(fn=run, inputs=[query_box], outputs=[output])
            query_box.submit(fn=run, inputs=[query_box], outputs=[output])

        # ── Tab 2: How it works ────────────────────────────────────────────
        with gr.Tab("⚙️ How It Works"):
            gr.Markdown("""
## Multi-Agent Pipeline

Each query passes through **four specialist AI agents** in sequence:

| Step | Agent | Role |
|------|-------|------|
| 1 | **Car Safety Agent** | Retrieves relevant facts from the SA car knowledge base (RAG) and assesses which vehicles suit local roads and conditions |
| 2 | **SA Market Agent** | Recommends specific models available in South Africa with ZAR pricing across three budget tiers |
| 3 | **Ownership Cost Agent** | Breaks down monthly fuel, insurance, maintenance, and licensing costs for each recommendation |
| 4 | **Final Recommendation Agent** | Synthesises all three agents into a clear report with a single top pick |

## RAG Knowledge Base

- Facts are loaded from `sa_cars.txt` and embedded using `all-MiniLM-L6-v2`
- Stored in a **ChromaDB** persistent vector database
- Top-6 most relevant facts are retrieved per query using cosine similarity

## Models Used

- **claude-haiku-4-5-20251001** — fast, cheap; used for agents 1, 2, and 3
- **claude-sonnet-4-6** — more capable; used for the final recommendation report
            """)

        
        with gr.Tab("📚 Knowledge Base"):
            gr.Markdown("### Facts currently in the SA Car knowledge base")
            with open("sa_cars.txt", encoding="utf-8") as f:
                kb_text = f.read()
            gr.Textbox(
                value=kb_text,
                label=f"sa_cars.txt ({collection.count()} facts indexed)",
                lines=20,
                interactive=False
            )
            gr.Markdown(
                "_To update: edit `sa_cars.txt` and re-run Cell 4 to rebuild the vector store._"
            )

    gr.HTML("""
    <div style="text-align:center; padding:1rem 0 0.5rem; color:#555; font-size:0.75rem;">
      SA Car Advisor · Multi-Agent RAG · Built with Anthropic Claude
    </div>
    """)


ui.launch(share=False, server_name="0.0.0.0", server_port=7860)